# Best Submission Pipeline

Clean archive of Daniel's best-performing pipeline for the final Kaggle submission. It combines a tuned classical TF-IDF ensemble with a fine-tuned Spanish biomedical-clinical transformer and writes blended submissions.


## 1. Setup and Paths

Imports, folder paths, and shared constants. Run this notebook from the `best_submission` folder. It reads shared data from the repository root `data/` folder and writes new submissions to `best_submission/output/`.


In [1]:
from pathlib import Path
import re, time, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.special import softmax

SUBMISSION_DIR = Path.cwd()
PROJECT_ROOT = SUBMISSION_DIR.parent
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
CODIESP_DIR = PROJECT_ROOT / 'data' / 'codiesp'
PRED_DIR = SUBMISSION_DIR / 'output' / 'predictions'
SUB_DIR = SUBMISSION_DIR / 'output' / 'submissions'
for p in [DATA_RAW_DIR, CODIESP_DIR, PRED_DIR, SUB_DIR]:
    p.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

## Optional Colab File Setup

Use this only when running in Colab and uploading files manually. In the repository archive, the expected files are already under the root `data/raw` and `data/codiesp` folders.


In [2]:
import shutil
from pathlib import Path

target = Path('/content/data/raw')
codiesp_target = Path('/content/data/codiesp')
target.mkdir(parents=True, exist_ok=True)
codiesp_target.mkdir(parents=True, exist_ok=True)

needed = ['codification_data.csv', 'leaderboard_data.csv', 'icd_d_p_pairs.csv']
for name in needed:
    if (target / name).exists():
        continue
    # search common Colab locations
    for base in ['/content', '/content/sample_data', '/content/drive/MyDrive']:
        src = Path(base) / name
        if src.exists():
            shutil.copy(src, target / name)
            break

found = [n for n in needed if (target / n).exists()]
print('ready in /content/data/raw:', found)
missing = set(needed) - set(found)
if missing:
    print('STILL MISSING:', missing, '— upload these to /content first')
# Optional: copy CodiEsp TSVs when running in Colab.
for src in list(Path('/content').glob('*annotations_task*_processed.tsv')):
    shutil.copy(src, codiesp_target / src.name)
print('CodiEsp TSVs in /content/data/codiesp:', len(list(codiesp_target.glob('*annotations_task*_processed.tsv'))))


ready in /content/data/raw: ['codification_data.csv', 'leaderboard_data.csv', 'icd_d_p_pairs.csv']


## 2. Text Cleaning and Classical Model Helpers

Defines the normalisation, TF-IDF feature union, probability alignment, and classical ensemble helper functions.


In [3]:
import re, unicodedata

_STOP_ES = set("de la el los las y o en a por con para del al un una unas unos su sus se le lo es son".split())

def _stem_es(w):
    for suf in ('ciones','cion','idades','idad','mente','ales','os','as','es','a','o','e','s'):
        if len(w) > 4 and w.endswith(suf):
            return w[:-len(suf)]
    return w

def normalize_text(text):
    """Accent-strip + lowercase + remove Spanish stopwords + light stemming. +0.38 val acc, measured."""
    if pd.isna(text):
        return ''
    t = re.sub(r'<[^>]+>', '', str(text)).replace('`', "'").lower()
    t = unicodedata.normalize('NFD', t)
    t = ''.join(c for c in t if unicodedata.category(c) != 'Mn')
    t = re.sub(r'[^a-z0-9 ]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return ' '.join(_stem_es(w) for w in t.split() if w not in _STOP_ES)


def build_features():
    # char_wb(2,5) measured best — 2-grams help abbreviations like VHC, TCE
    return FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ])


def align(P, classes, master):
    idx = {c: i for i, c in enumerate(classes)}
    out = np.zeros((P.shape[0], len(master)))
    for j, c in enumerate(master):
        if c in idx:
            out[:, j] = P[:, idx[c]]
    return out


def svc_proba(svc, X):
    d = svc.decision_function(X)
    if d.ndim == 1:
        d = np.vstack([-d, d]).T
    return softmax(d, axis=1)


def fit_ensemble(texts, labels, C_svc=0.25, C_lr=3.0):
    vec = build_features()
    X = vec.fit_transform(texts)
    svc = LinearSVC(C=C_svc, max_iter=6000, random_state=RANDOM_STATE).fit(X, labels)
    nb = ComplementNB().fit(X, labels)
    lr = LogisticRegression(C=C_lr, max_iter=1500, random_state=RANDOM_STATE).fit(X, labels)
    return vec, svc, nb, lr


def ensemble_proba(vec, svc, nb, lr, texts, master, weights):
    X = vec.transform(texts)
    w_svc, w_nb, w_lr = weights
    return (w_svc * align(svc_proba(svc, X), svc.classes_, master)
            + w_nb * align(nb.predict_proba(X), nb.classes_, master)
            + w_lr * align(lr.predict_proba(X), lr.classes_, master))

## 3. Load Competition Data

Loads the ASHO/UAB training literals and leaderboard literals, creates the chapter target, and prints quick dataset diagnostics.


In [4]:
train = pd.read_csv(DATA_RAW_DIR / 'codification_data.csv')
test = pd.read_csv(DATA_RAW_DIR / 'leaderboard_data.csv')

train['Literal_norm'] = train['Literal'].apply(normalize_text)
test['Literal_norm'] = test['Literal'].apply(normalize_text)
train['chapter'] = train['Code'].astype(str).str[0]

master = sorted(set(train['chapter']))
master_arr = np.array(master)

print(f'train rows: {len(train):,} | test rows: {len(test):,} | chapters: {len(master)}')
overlap = test['Literal_norm'].isin(set(train['Literal_norm'])).mean()
print(f'leaderboard literals with an exact match in training: {overlap*100:.1f}%')
print('top chapters:', train['chapter'].value_counts().head(8).to_dict())

train rows: 13,700 | test rows: 6,667 | chapters: 36
leaderboard literals with an exact match in training: 65.6%
top chapters: {'Z': 1715, 'O': 1505, '0': 1141, '6': 637, '3': 592, 'B': 579, 'N': 536, 'E': 500}


## 4. Tune the Classical Ensemble

Compares the previous single-SVC baseline against the improved TF-IDF ensemble and tunes the SVC regularisation/ensemble weights on a validation split.


In [5]:
t0 = time.time()
tr, va = train_test_split(train, test_size=0.25, random_state=RANDOM_STATE, stratify=train['chapter'])
yv = va['chapter'].values
Xva = va['Literal_norm'].fillna('').tolist()

# Baseline = original model's config: single LinearSVC, char(3,5)+word(1,2), C=1
base_vec = FeatureUnion([
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
    ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True))])
Xb = base_vec.fit_transform(tr['Literal_norm'])
base_svc = LinearSVC(C=1.0, max_iter=4000, random_state=RANDOM_STATE).fit(Xb, tr['chapter'].values)
base_acc = (base_svc.classes_[base_svc.decision_function(base_vec.transform(Xva)).argmax(1)] == yv).mean()

# New model: tune C, then ensemble weights
best_C, best_C_acc = 0.25, -1.0
for C in [0.25, 0.5, 1.0]:
    v, s, _, _ = fit_ensemble(tr['Literal_norm'], tr['chapter'].values, C_svc=C)
    acc = (s.classes_[svc_proba(s, v.transform(Xva)).argmax(1)] == yv).mean()
    if acc > best_C_acc:
        best_C_acc, best_C = acc, C

vec, svc, nb, lr = fit_ensemble(tr['Literal_norm'], tr['chapter'].values, C_svc=best_C)
Xt = vec.transform(Xva)
pS = align(svc_proba(svc, Xt), svc.classes_, master)
pN = align(nb.predict_proba(Xt), nb.classes_, master)
pL = align(lr.predict_proba(Xt), lr.classes_, master)
best_w, best_w_acc = (1.0, 0.3, 0.3), -1.0
for wn in [0.0, 0.3, 0.6]:
    for wl in [0.3, 0.6, 1.0]:
        acc = (master_arr[(pS + wn * pN + wl * pL).argmax(1)] == yv).mean()
        if acc > best_w_acc:
            best_w_acc, best_w = acc, (1.0, wn, wl)

print(f'tuned C = {best_C} | tuned ensemble weights (svc, nb, lr) = {best_w}')
print('-' * 50)
print(f'Baseline single SVC (original config) : {base_acc*100:5.2f}%')
print(f'New ensemble                          : {best_w_acc*100:5.2f}%')
print(f'Expected lift                         : {(best_w_acc-base_acc)*100:+.2f} pts')
print(f'(validation finished in {time.time()-t0:.0f}s)')

tuned C = 0.5 | tuned ensemble weights (svc, nb, lr) = (1.0, 0.6, 0.3)
--------------------------------------------------
Baseline single SVC (original config) : 54.69%
New ensemble                          : 55.36%
Expected lift                         : +0.67 pts
(validation finished in 119s)


## 5. Train Classical Model on the Full Dataset

Retrains the classical ensemble on all labelled examples, saves its leaderboard probabilities, and writes a standalone classical submission.


In [6]:
vec, svc, nb, lr = fit_ensemble(train['Literal_norm'], train['chapter'].values, C_svc=best_C)
ens_prob = ensemble_proba(vec, svc, nb, lr, test['Literal_norm'].fillna('').tolist(), master, best_w)
test['y_category'] = master_arr[ens_prob.argmax(1)]

submission = test[['id', 'Literal', 'y_category']].copy()
submission['y_category'] = submission['y_category'].replace('', 'null').fillna('null')

# saved so the transformer notebook can blend with it
np.save(PRED_DIR / 'classical_ens_prob.npy', ens_prob)
np.save(PRED_DIR / 'classical_master.npy', master_arr)

sub_path = SUB_DIR / 'classical_submission.csv'
submission.to_csv(sub_path, index=False)

assert list(submission.columns)[0] == 'id'
assert submission['y_category'].notna().all() and (submission['y_category'] != '').all()
assert len(submission) == len(test)
print('written:', sub_path, '| shape', submission.shape)
print('prediction distribution:', submission['y_category'].value_counts().head(10).to_dict())
submission.head(10)

written: /content/output/submissions/classical_submission.csv | shape (6667, 3)
prediction distribution: {'O': 928, 'Z': 904, '0': 693, 'B': 324, 'N': 288, '3': 282, 'E': 271, 'K': 243, 'I': 240, 'D': 204}


,id,Literal,y_category
0,1,AMNIODRENAJE,7
1,2,Hiperparatiroidismo primario,E
2,3,MIGRANYA parto,O
3,4,VHC,B
4,5,Absceso mama izq,N
5,6,miomas parto,D
6,7,cos estrany,9
7,8,TCE,8
8,9,osteogénesis imperfecta,Q
9,10,Soplo sistólico,R


## Transformer Augmentation

The first half of the notebook builds the classical ensemble. The second half adds a clinical transformer and blends the two probability distributions.


## 6. Install Transformer Dependencies

Colab-oriented install cell for the Hugging Face transformer stack.


In [7]:
%pip install -q -U "datasets>=2.19" "pyarrow>=15.0.0" "transformers>=4.40" "accelerate>=0.30"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
bigframes 2.40.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.


## 7. Transformer Configuration

Defines the Spanish biomedical-clinical RoBERTa model and training hyperparameters.


In [8]:
MODEL_NAME = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"   # Spanish biomedical-clinical
# Alternatives — re-run with each, keep the higher val acc:
#   "xlm-roberta-base"                   # multilingual, best Catalan coverage
#   "projecte-aina/roberta-base-ca-v2"   # Catalan, if Catalan dominates errors
EPOCHS = 5                            # a bit more, since XLM-R is larger
MAX_LEN = 48
LR      = 2e-5
BATCH   = 32
AUGMENT_WITH_REFERENCE = False
RANDOM_STATE = 42

## 8. Rebuild Local Data and Classical Probabilities for Blending

Recreates the cleaned training/test views and helper functions used by the transformer/blend stage.


In [9]:
from pathlib import Path
import re, unicodedata, time
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.special import softmax

SUBMISSION_DIR = Path.cwd()
PROJECT_ROOT = SUBMISSION_DIR.parent
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
CODIESP_DIR = PROJECT_ROOT / 'data' / 'codiesp'
SUB_DIR = SUBMISSION_DIR / 'output' / 'submissions'; SUB_DIR.mkdir(parents=True, exist_ok=True)

def normalize_text(t):
    if pd.isna(t): return ''
    t = re.sub(r'<[^>]+>', '', str(t)).replace('`', "'").lower()
    t = unicodedata.normalize('NFD', t); t = ''.join(c for c in t if unicodedata.category(c) != 'Mn')
    t = re.sub(r'[^a-z0-9 ]', ' ', t); return re.sub(r'\s+', ' ', t).strip()

def light_clean(t):   # transformer keeps accents/case
    if pd.isna(t): return ''
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', '', str(t))).strip()

train = pd.read_csv(DATA_RAW_DIR / 'codification_data.csv')
test  = pd.read_csv(DATA_RAW_DIR / 'leaderboard_data.csv')
train['chapter'] = train['Code'].astype(str).str[0]
train['ln'] = train['Literal'].apply(normalize_text); test['ln'] = test['Literal'].apply(normalize_text)
train['lc'] = train['Literal'].apply(light_clean);    test['lc'] = test['Literal'].apply(light_clean)

CHAPTERS = sorted(set(train['chapter']))
c2i = {c: i for i, c in enumerate(CHAPTERS)}; i2c = {i: c for c, i in c2i.items()}
marr = np.array(CHAPTERS)

tr, va = train_test_split(train, test_size=0.25, random_state=RANDOM_STATE, stratify=train['chapter'])
yv = va['chapter'].values

def _align(P, classes):
    idx = {c: i for i, c in enumerate(classes)}; out = np.zeros((P.shape[0], len(CHAPTERS)))
    for j, c in enumerate(CHAPTERS):
        if c in idx: out[:, j] = P[:, idx[c]]
    return out

def classical_probs(train_texts, train_labels, eval_texts):
    vec = FeatureUnion([('c', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
                        ('w', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True))])
    X = vec.fit_transform(train_texts); Xe = vec.transform(eval_texts)
    s = LinearSVC(C=0.25, max_iter=6000, random_state=RANDOM_STATE).fit(X, train_labels)
    n = ComplementNB().fit(X, train_labels)
    l = LogisticRegression(C=3.0, max_iter=1500, random_state=RANDOM_STATE).fit(X, train_labels)
    sd = s.decision_function(Xe)
    pS = softmax(sd, axis=1) if sd.ndim > 1 else softmax(np.vstack([-sd, sd]).T, axis=1)
    return 1.0*_align(pS, s.classes_) + 0.3*_align(n.predict_proba(Xe), n.classes_) + 0.3*_align(l.predict_proba(Xe), l.classes_)

t0 = time.time()
cl_val = classical_probs(tr['ln'], tr['chapter'].values, va['ln'].fillna('').tolist())
cl_lb  = classical_probs(train['ln'], train['chapter'].values, test['ln'].fillna('').tolist())
print(f"classical ready in {time.time()-t0:.0f}s | val acc {(marr[cl_val.argmax(1)]==yv).mean()*100:.2f}%")

classical ready in 99s | val acc 54.92%


## Dependency Pin for Colab

Pins `pyarrow`/`datasets` versions if Colab has a binary mismatch. Restart the runtime after this cell if Colab asks for it.


In [10]:
!pip install -q --force-reinstall "pyarrow==17.0.0" "datasets==3.0.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
bigframes 2.

## 9. Add External CodiEsp Training Pairs

Parses the CodiEsp TSV files and converts them into extra chapter-labelled Spanish clinical examples.


In [11]:
import pandas as pd, re, unicodedata, glob

def _fix_enc(s):
    # CodiEsp text is latin-1 bytes mis-decoded as utf-8; reverse the mojibake
    if not isinstance(s, str): return ''
    try:
        return s.encode('latin-1').decode('utf-8')
    except Exception:
        return s

# parse all 6 CodiEsp TSVs (task1/2 = 4 cols, task3 = 5 cols; we take docid,type,code,literal)
_codiesp_files = list(CODIESP_DIR.glob('*annotations_task*_processed.tsv'))
if not _codiesp_files:
    _codiesp_files = [Path(f) for f in glob.glob('/content/*annotations_task*_processed.tsv')]
if not _codiesp_files:
    _codiesp_files = [Path(f) for f in glob.glob('/content/data/codiesp/*annotations_task*_processed.tsv')]
_rows = []
for _f in _codiesp_files:
    with open(_f, encoding='utf-8') as _fh:
        for _line in _fh:
            _p = _line.rstrip('\n').split('\t')
            if len(_p) < 4:
                continue
            _code = _p[2].replace('(automatic suggestion)', '').strip()
            _lit  = _fix_enc(_p[3].strip())
            if _code and _lit:
                _rows.append((_lit, _code))

codiesp_extra = pd.DataFrame(_rows, columns=['Literal', 'Code']).drop_duplicates()
# chapter = first char of code, UPPERCASE to match the competition's codes
codiesp_extra['chapter'] = codiesp_extra['Code'].astype(str).str[0].str.upper()
# clean text for the transformer (keep accents/case, just tidy)
codiesp_extra['lc'] = codiesp_extra['Literal'].apply(
    lambda t: re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', '', str(t))).strip())
# keep only chapters that exist in the competition label space
codiesp_extra = codiesp_extra[codiesp_extra['chapter'].isin(c2i)].drop_duplicates('lc')

print('CodiEsp files parsed:', len(_codiesp_files))
print(f'CodiEsp extra training pairs: {len(codiesp_extra):,}')
print('chapters added:', sorted(codiesp_extra['chapter'].unique()))

CodiEsp extra training pairs: 6,362
chapters added: ['0', '1', '2', '3', '4', '5', '6', '8', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'X', 'Y', 'Z']


## 10. Fine-Tune the Clinical Transformer

Fine-tunes `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es` with class weighting, using the competition literals plus CodiEsp examples.


In [12]:
import torch
import numpy as np
import pandas as pd
import re
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| model:', MODEL_NAME)

# --- tokenizer + tok_fn (defined here so the cell is self-contained) ---
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
def tok_fn(b):
    return tok(b['text'], truncation=True, max_length=MAX_LEN)

# --- train/val split (same seed -> same split as before) ---
tr, va = train_test_split(train, test_size=0.25, random_state=RANDOM_STATE,
                          stratify=train['chapter'])
yv = va['chapter'].values

# --- NEW: combine literals with the ICD reference dictionary ---
def light_clean(t):
    if pd.isna(t): return ''
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', '', str(t))).strip()

ref = pd.read_csv(DATA_RAW_DIR / 'icd_d_p_pairs.csv')
ref['chapter'] = ref['Code'].astype(str).str[0]
ref = ref[ref['chapter'].isin(c2i)].copy()
ref_sample = ref.sample(n=min(40000, len(ref)), random_state=RANDOM_STATE)

# --- training data: your literals + CodiEsp pairs (the new bilingual signal) ---
tr_text = tr['lc'].tolist() + codiesp_extra['lc'].tolist()
tr_lab  = tr['chapter'].map(c2i).tolist() + codiesp_extra['chapter'].map(c2i).tolist()
ds_tr = Dataset.from_dict({'text': tr_text, 'labels': tr_lab}).map(tok_fn, batched=True)
print(f'training on {len(tr_text):,} examples ({len(tr)} literals + {len(codiesp_extra)} CodiEsp)')

# --- class weights (counter chapter imbalance) ---
cw = compute_class_weight('balanced', classes=np.arange(len(CHAPTERS)), y=np.array(tr_lab))
class_weights = torch.tensor(cw, dtype=torch.float)

# --- custom trainer ---
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            out.logits, labels, weight=class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

# --- probability matrix from a trained trainer, in CHAPTERS order ---
def probs_from(trainer_obj, texts):
    ds = Dataset.from_dict({'text': list(texts)}).map(tok_fn, batched=True)
    logits = trainer_obj.predict(ds).predictions
    return torch.softmax(torch.tensor(logits), dim=1).numpy()

# --- single run (averaging didn't help; new data is the lever) ---
SEEDS = [42]
tr_val_runs, tr_lb_runs = [], []

for sd in SEEDS:
    print(f'\n===== training seed {sd} =====')
    torch.manual_seed(sd); np.random.seed(sd)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CHAPTERS), id2label=i2c, label2id=c2i)

    args = TrainingArguments(
        output_dir=f'output/hf_ref_seed{sd}',
        per_device_train_batch_size=BATCH,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.06,
        logging_steps=200,
        save_strategy='no',
        report_to='none',
        fp16=(device == 'cuda'),
        seed=sd,
    )

    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds_tr,
        data_collator=DataCollatorWithPadding(tok))
    trainer.train()

    tr_val_runs.append(probs_from(trainer, va['lc'].tolist()))
    tr_lb_runs.append(probs_from(trainer, test['lc'].tolist()))

    del model, trainer
    torch.cuda.empty_cache()

tr_val = np.mean(tr_val_runs, axis=0)
tr_lb  = np.mean(tr_lb_runs, axis=0)
print(f'transformer (with reference data) val acc : {(marr[tr_val.argmax(1)]==yv).mean()*100:.2f}%')

device: cuda | model: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/16637 [00:00<?, ? examples/s]

training on 16,637 examples (10275 literals + 6362 CodiEsp)

===== training seed 42 =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `w

Step,Training Loss
200,3.514761
400,2.771442
600,2.169294
800,1.871653
1000,1.689661
1200,1.491444
1400,1.325139
1600,1.314260
1800,1.171388
2000,1.141509


Map:   0%|          | 0/3425 [00:00<?, ? examples/s]

Map:   0%|          | 0/6667 [00:00<?, ? examples/s]

transformer (with reference data) val acc : 49.78%


## 11. Blend and Write Kaggle Submissions

Combines transformer and classical probabilities with several alpha values. The best archived submission is `codi_a20.csv`.


In [13]:
import numpy as np

# show how the CodiEsp-trained transformer did
print(f'transformer (CodiEsp) val acc : {(marr[tr_val.argmax(1)]==yv).mean()*100:.2f}%')
print(f'classical val acc             : {(marr[cl_val.argmax(1)]==yv).mean()*100:.2f}%')

def write_alpha(a):
    prob = a * tr_lb + (1 - a) * cl_lb
    out = test[['id', 'Literal']].copy()
    out['y_category'] = marr[prob.argmax(1)].astype(str)
    out['y_category'] = out['y_category'].replace('', 'null').fillna('null')
    name = f'codi_a{int(round(a*100)):02d}.csv'
    out.to_csv(SUB_DIR / name, index=False)
    print('written:', name)

for a in [0.20, 0.24, 0.28]:
    write_alpha(a)

transformer (CodiEsp) val acc : 49.78%
classical val acc             : 54.92%
written: codi_a20.csv
written: codi_a24.csv
written: codi_a28.csv


## 12. Final Sanity Checks

Quick checks that the external data and transformer probabilities exist before using the submission outputs.


In [14]:
print('codiesp_extra:', len(codiesp_extra), 'pairs')
print('tr_lb exists:', 'tr_lb' in dir())

codiesp_extra: 6362 pairs
tr_lb exists: True
